## Handoof Multi-Agent Design Pattern with LangChain

![Handoff.png](./Images/Handoff.png)

### Installing Utilities and Libraries

In [ ]:
%pip install -U \
    databricks-langchain==0.20.0 \
    langgraph==1.2.11 \
    mlflow

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setup MLflow Tracing

In [ ]:
import mlflow
import os
import os
from dotenv import load_dotenv
from typing import Literal
from langchain.agents import AgentState, create_agent
from langchain.messages import AIMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import NotRequired

# Enable auto-tracing for OpenAI
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Handoff-trace")

### Instantiate the ChatDatabricks Class

In [ ]:
import json
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0.1,
    max_tokens=20000,
)

### Create the Agent State Class

In [ ]:
# Define state with active_agent tracker
class MultiAgentState(AgentState):
    active_agent: NotRequired[str]

### Create the Sales Agent Transfer Tool

In [ ]:
from mlflow.entities import SpanType

@tool
@mlflow.trace(name="transfer_to_sales", span_type=SpanType.TOOL)
def transfer_to_sales(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to the sales agent"""
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )

    transfer_message = ToolMessage(
        content = "Transferred to sales agent from support agent",
        tool_call_id = runtime.tool_call_id
    )

    return Command(
        goto="sales_agent",
        update = {
            "active_agent": "sales_agent",
            "messages": [last_ai_message, transfer_message]
        },
        graph = Command.PARENT
    )

### Create the Support Agent Transfer Tool

In [ ]:
@tool
@mlflow.trace(name="transfer_to_support", span_type=SpanType.TOOL)
def transfer_to_support(
    runtime: ToolRuntime,
) -> Command:
    """Transfer to the support agent."""
    last_ai_message = next(
        msg for msg in reversed(runtime.state["messages"]) if isinstance(msg, AIMessage)
    )
    transfer_message = ToolMessage(
        content="Transferred to support agent from sales agent",
        tool_call_id=runtime.tool_call_id,
    )
    return Command(
        goto="support_agent",
        update={
            "active_agent": "support_agent",
            "messages": [last_ai_message, transfer_message],
        },
        graph=Command.PARENT,
    )

### Create Agents with the Handoff Tools

In [ ]:
# Create the sales agent
sales_agent = create_agent(
    model = model,
    tools = [transfer_to_support],
    system_prompt = "You are a sales agent. Help with sales inquiries. If asked about technical issues or support, transfer to the support agent."
)

# Create the support agent
support_agent = create_agent(
    model=model,
    tools=[transfer_to_sales],
    system_prompt="You are a support agent. Help with technical issues. If asked about pricing or purchasing, transfer to the sales agent.",
)

### Create Agent Nodes that invoke the Agents

In [ ]:
# Create the sales agent node
def call_sales_agent(state: MultiAgentState) -> Command:
    """Node that calls the sales agent."""
    response = sales_agent.invoke(state)
    return response

# Create the support agent node
def call_support_agent(state: MultiAgentState) -> Command:
    """Node that calls the support agent."""
    response = support_agent.invoke(state)
    return response

### Create the Workflow Routers

In [ ]:
# Create router that checks if we should end or continue
def route_after_agent(
    state: MultiAgentState,
) -> Literal["sales_agent", "support_agent", "__end__"]:
    """Route based on active_agent, or END if the agent finished without handoff."""
    messages = state.get("messages", [])

    # Check the last message - if it's an AIMessage without tool calls, we're done
    if messages:
        last_msg = messages[-1]
        if isinstance(last_msg, AIMessage) and not last_msg.tool_calls:
            return "__end__"

    # Otherwise route to the active agent
    active = state.get("active_agent", "sales_agent")
    return active if active else "sales_agent"

# Create the Initial Router
def route_initial(
    state: MultiAgentState,
) -> Literal["sales_agent", "support_agent"]:
    """Route to the active agent based on state, default to sales agent."""
    return state.get("active_agent") or "sales_agent"

### Build the Workflow Graph

In [ ]:
# Build the graph
builder = StateGraph(MultiAgentState)
builder.add_node("sales_agent", call_sales_agent)
builder.add_node("support_agent", call_support_agent)

# Start with conditional routing based on initial active_agent
builder.add_conditional_edges(START, route_initial, ["sales_agent", "support_agent"])

# After each agent, check if we should end or route to another agent
builder.add_conditional_edges(
    "sales_agent", route_after_agent, ["sales_agent", "support_agent", END]
)
builder.add_conditional_edges(
    "support_agent", route_after_agent, ["sales_agent", "support_agent", END]
)

graph = builder.compile()

### Generate the Mermaid Diagram of the Workflow

In [ ]:
png = graph.get_graph().draw_mermaid_png()

with open("handoff_workflow.png", "wb") as f:
    f.write(png)

### Invoke the Workflow

In [ ]:
from langchain_core.messages import AIMessage

@mlflow.trace(name="multi_agent_chat")
def run_multi_agent_chat(user_input: str, state: dict):

    # Add user message to conversation
    state["messages"].append(
        {
            "role": "user",
            "content": user_input,
        }
    )

    # Run the multi-agent graph
    result = graph.invoke(state)

    # Find the latest AI response
    last_ai_message = next(
        msg
        for msg in reversed(result["messages"])
        if isinstance(msg, AIMessage)
    )

    # Print the graph messages
    for msg in result["messages"]:
        msg.pretty_print()

    return result, last_ai_message.content

In [ ]:
state = {
    "messages": [],
    "active_agent": "sales_agent",
}

print("Multi-Agent Chat")
print("Type 'exit' to end the conversation.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chat ended.")
        break

    state, response = run_multi_agent_chat(
        user_input,
        state
    )